# tiny transformer from scratch

building the smallest plausible decoder-only transformer in pytorch. tokens are characters of a tiny shakespeare-ish string. just to see the shapes line up. proper version will be its own repo in 2022.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MHSA(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.qkv = nn.Linear(d, 3*d)
        self.proj = nn.Linear(d, d)
        self.n_heads = n_heads
    def forward(self, x, mask):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        H = self.n_heads
        q = q.view(B, T, H, D//H).transpose(1, 2)
        k = k.view(B, T, H, D//H).transpose(1, 2)
        v = v.view(B, T, H, D//H).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(D//H)
        att = att.masked_fill(mask == 0, -1e9)
        att = F.softmax(att, dim=-1)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, D)
        return self.proj(y)